In [0]:
from pyspark.sql import functions as F

REFERENCE_DATE = "2026-08-01"

N_CUSTOMERS = 2_000
N_CONTRACTS = 5_000

BASE_PATH = "/Volumes/credlake/landing/raw"
SOURCE_BATCH_ID = f"initial_{REFERENCE_DATE}"

print(f"Data de referência: {REFERENCE_DATE}")
print(f"Diretório de destino: {BASE_PATH}")

In [0]:
# Produtos Financeiros sintéticos
product_rows = [
    (1, "Crédito Pessoal", "PF", 0.2890, 48),
    (2, "Consignado", "PF", 0.1890, 60),
    (3, "Financiamento", "PF", 0.1590, 60),
    (4, "Capital de Giro", "PJ", 0.2490, 36),
    (5, "Crédito Rural", "PF/PJ", 0.1290, 48),
]

products = (
    spark.createDataFrame(
        product_rows,
        [
            "product_id",
            "product_name",
            "allowed_customer_type",
            "base_annual_interest_rate",
            "max_term_months",
        ],
    )
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("max_term_months", F.col("max_term_months").cast("int"))
    .withColumn("source_batch_id", F.lit(SOURCE_BATCH_ID))
)

# Clientes sintéticos
states = [
    "RO", "AC", "AM", "RR", "PA", "AP", "TO",
    "MT", "MS", "GO", "DF", "SP", "RJ", "MG",
    "PR", "SC", "RS", "BA", "PE", "CE"
]

ratings = ["A", "B", "C", "D", "E"]

state_array = F.array(*[F.lit(state) for state in states])
rating_array = F.array(*[F.lit(rating) for rating in ratings])

customers = (
    spark.range(1, N_CUSTOMERS + 1)
    .withColumnRenamed("id", "customer_id")

    .withColumn(
        "customer_type",
        F.when(F.pmod(F.col("customer_id"), F.lit(5)) == 0, F.lit("PJ"))
         .otherwise(F.lit("PF"))
    )

    .withColumn(
        "customer_name",
        F.concat(
            F.lit("Cliente Sintético "),
            F.lpad(F.col("customer_id").cast("string"), 6, "0")
        )
    )

    .withColumn(
        "document_id",
        F.concat(
            F.lit("SYN"),
            F.lpad(F.col("customer_id").cast("string"), 11, "0")
        )
    )

    .withColumn(
        "email",
        F.concat(
            F.lit("cliente"),
            F.lpad(F.col("customer_id").cast("string"), 6, "0"),
            F.lit("@example.invalid")
        )
    )

    .withColumn(
        "state",
        F.element_at(
            state_array,
            (
                F.pmod(
                    F.xxhash64(F.col("customer_id")),
                    F.lit(len(states))
                ) + 1
            ).cast("int")
        )
    )

    .withColumn(
        "risk_rating",
        F.element_at(
            rating_array,
            (
                F.pmod(
                    F.xxhash64(F.col("customer_id"), F.lit("risk")),
                    F.lit(len(ratings))
                ) + 1
            ).cast("int")
        )
    )

    .withColumn(
        "monthly_income",
        F.when(
            F.col("customer_type") == "PJ",
            (
                F.lit(20_000) +
                F.pmod(
                    F.xxhash64(F.col("customer_id"), F.lit("income")),
                    F.lit(480_000)
                )
            ).cast("double")
        ).otherwise(
            (
                F.lit(1_500) +
                F.pmod(
                    F.xxhash64(F.col("customer_id"), F.lit("income")),
                    F.lit(28_500)
                )
            ).cast("double")
        )
    )

    .withColumn(
        "created_at",
        F.date_sub(
            F.to_date(F.lit(REFERENCE_DATE)),
            (
                F.pmod(
                    F.xxhash64(F.col("customer_id"), F.lit("created")),
                    F.lit(730)
                ) + 30
            ).cast("int")
        )
    )

    .withColumn(
        "updated_at",
        F.to_timestamp(F.lit(f"{REFERENCE_DATE} 06:00:00"))
    )

    .withColumn("is_active", F.lit(True))
    .withColumn("source_system", F.lit("CUSTOMER_CORE"))
    .withColumn("source_batch_id", F.lit(SOURCE_BATCH_ID))
)

display(customers.limit(10))

In [0]:
# Contratos válidos
term_options = F.array(
    F.lit(6),
    F.lit(12),
    F.lit(18),
    F.lit(24),
    F.lit(36),
    F.lit(48),
    F.lit(60)
)

contracts_base = (
    spark.range(1, N_CONTRACTS + 1)
    .withColumnRenamed("id", "contract_id")

    .withColumn(
        "customer_id",
        (
            F.pmod(
                F.xxhash64(F.col("contract_id"), F.lit("customer")),
                F.lit(N_CUSTOMERS)
            ) + 1
        ).cast("long")
    )

    .withColumn(
        "product_id",
        (
            F.pmod(
                F.xxhash64(F.col("contract_id"), F.lit("product")),
                F.lit(5)
            ) + 1
        ).cast("int")
    )

    .withColumn(
        "contract_date",
        F.date_sub(
            F.to_date(F.lit(REFERENCE_DATE)),
            (
                F.pmod(
                    F.xxhash64(F.col("contract_id"), F.lit("date")),
                    F.lit(540)
                ) + 30
            ).cast("int")
        )
    )

    .withColumn(
        "principal_amount",
        (
            F.lit(1_000) +
            F.pmod(
                F.xxhash64(F.col("contract_id"), F.lit("principal")),
                F.lit(199_000)
            )
        ).cast("double")
    )

    .withColumn(
        "term_months",
        F.element_at(
            term_options,
            (
                F.pmod(
                    F.xxhash64(F.col("contract_id"), F.lit("term")),
                    F.lit(7)
                ) + 1
            ).cast("int")
        )
    )
)

risk_premium = (
    F.when(F.col("risk_rating") == "A", F.lit(0.000))
     .when(F.col("risk_rating") == "B", F.lit(0.015))
     .when(F.col("risk_rating") == "C", F.lit(0.035))
     .when(F.col("risk_rating") == "D", F.lit(0.065))
     .otherwise(F.lit(0.100))
)

contracts = (
    contracts_base

    .join(
        customers.select("customer_id", "risk_rating"),
        on="customer_id",
        how="left"
    )

    .join(
        products.select(
            "product_id",
            "base_annual_interest_rate"
        ),
        on="product_id",
        how="left"
    )

    .withColumn(
        "annual_interest_rate",
        F.round(
            F.col("base_annual_interest_rate") + risk_premium,
            4
        )
    )

    .withColumn(
        "contract_status",
        F.when(
            F.pmod(
                F.xxhash64(F.col("contract_id"), F.lit("status")),
                F.lit(100)
            ) < 8,
            F.lit("CLOSED")
        ).otherwise(F.lit("ACTIVE"))
    )

    .withColumn(
        "source_updated_at",
        F.to_timestamp(F.lit(f"{REFERENCE_DATE} 07:00:00"))
    )

    .withColumn("source_system", F.lit("CREDIT_CORE"))
    .withColumn("source_batch_id", F.lit(SOURCE_BATCH_ID))

    .select(
        "contract_id",
        "customer_id",
        "product_id",
        "contract_date",
        "principal_amount",
        "annual_interest_rate",
        "term_months",
        "contract_status",
        "source_updated_at",
        "source_system",
        "source_batch_id"
    )
)

# Problemas propositais na origem
duplicate_contract = contracts.filter(
    F.col("contract_id") == 10
)

negative_contract = (
    contracts
    .filter(F.col("contract_id") == 11)
    .withColumn("contract_id", F.lit(9_000_001).cast("long"))
    .withColumn("principal_amount", F.lit(-500.00))
)

orphan_contract = (
    contracts
    .filter(F.col("contract_id") == 12)
    .withColumn("contract_id", F.lit(9_000_002).cast("long"))
    .withColumn("customer_id", F.lit(9_999_999).cast("long"))
)

contracts_source = (
    contracts
    .unionByName(duplicate_contract)
    .unionByName(negative_contract)
    .unionByName(orphan_contract)
)

# Parcelas
installments = (
    contracts

    .withColumn(
        "installment_number",
        F.explode(
            F.sequence(
                F.lit(1),
                F.col("term_months")
            )
        )
    )

    .withColumn(
        "installment_id",
        F.concat_ws(
            "-",
            F.col("contract_id"),
            F.lpad(
                F.col("installment_number").cast("string"),
                3,
                "0"
            )
        )
    )

    .withColumn(
        "due_date",
        F.expr(
            "add_months(contract_date, installment_number)"
        )
    )

    .withColumn(
        "scheduled_amount",
        F.round(
            (
                F.col("principal_amount") *
                (
                    F.lit(1) +
                    F.col("annual_interest_rate") *
                    F.col("term_months") / F.lit(12)
                )
            ) / F.col("term_months"),
            2
        )
    )

    .withColumn("source_system", F.lit("INSTALLMENT_CORE"))
    .withColumn("source_batch_id", F.lit(SOURCE_BATCH_ID))

    .select(
        "installment_id",
        "contract_id",
        "installment_number",
        "due_date",
        "scheduled_amount",
        "source_system",
        "source_batch_id"
    )
)

# Pagamentos
payments = (
    installments

    .filter(
        F.col("due_date") <= F.to_date(F.lit(REFERENCE_DATE))
    )

    .filter(
        F.pmod(
            F.xxhash64(F.col("installment_id"), F.lit("paid")),
            F.lit(100)
        ) < 82
    )

    .withColumn(
        "delay_days",
        (
            F.pmod(
                F.xxhash64(F.col("installment_id"), F.lit("delay")),
                F.lit(55)
            ) - 5
        ).cast("int")
    )

    .withColumn(
        "payment_date",
        F.date_add(
            F.col("due_date"),
            F.col("delay_days")
        )
    )

    .withColumn(
        "amount_paid",
        F.when(
            F.pmod(
                F.xxhash64(F.col("installment_id"), F.lit("partial")),
                F.lit(100)
            ) < 12,
            F.round(F.col("scheduled_amount") * F.lit(0.60), 2)
        ).otherwise(F.col("scheduled_amount"))
    )

    .withColumn(
        "payment_id",
        F.sha2(
            F.concat_ws(
                "|",
                F.col("installment_id"),
                F.col("payment_date"),
                F.col("amount_paid")
            ),
            256
        )
    )

    .withColumn("event_type", F.lit("PAYMENT"))

    .withColumn(
        "event_timestamp",
        F.to_timestamp(
            F.concat(
                F.date_format(F.col("payment_date"), "yyyy-MM-dd"),
                F.lit(" 10:00:00")
            )
        )
    )

    .withColumn("source_system", F.lit("PAYMENT_GATEWAY"))
    .withColumn("source_batch_id", F.lit(SOURCE_BATCH_ID))

    .select(
        "payment_id",
        "installment_id",
        "contract_id",
        "installment_number",
        "payment_date",
        "amount_paid",
        "event_type",
        "event_timestamp",
        "source_system",
        "source_batch_id"
    )
)

# Duplicidade proposital
duplicate_payment = payments.limit(1)

# Pagamento com valor nulo proposital
invalid_payment = (
    payments
    .limit(1)
    .withColumn("payment_id", F.lit("INVALID_NULL_AMOUNT"))
    .withColumn("amount_paid", F.lit(None).cast("double"))
)

payments_source = (
    payments
    .unionByName(duplicate_payment)
    .unionByName(invalid_payment)
)

display(contracts_source.limit(10))

In [0]:
# Caminhos dos arquivos de origem
products_path = (
    f"{BASE_PATH}/products/full"
)

customers_path = (
    f"{BASE_PATH}/customers/snapshot_date={REFERENCE_DATE}"
)

contracts_path = (
    f"{BASE_PATH}/contracts/batch_date={REFERENCE_DATE}"
)

installments_path = (
    f"{BASE_PATH}/installments/snapshot_date={REFERENCE_DATE}"
)

payments_path = (
    f"{BASE_PATH}/payments/event_date={REFERENCE_DATE}"
)

# Gravação
(
    products
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(products_path)
)

(
    customers
    .coalesce(2)
    .write
    .mode("overwrite")
    .json(customers_path)
)

(
    contracts_source
    .coalesce(2)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(contracts_path)
)

(
    installments
    .repartition(4)
    .write
    .mode("overwrite")
    .parquet(installments_path)
)

(
    payments_source
    .repartition(4)
    .write
    .mode("overwrite")
    .json(payments_path)
)

# Resumo da geração
generation_summary = spark.createDataFrame(
    [
        ("products", products.count(), products_path),
        ("customers", customers.count(), customers_path),
        ("contracts_source", contracts_source.count(), contracts_path),
        ("installments", installments.count(), installments_path),
        ("payments_source", payments_source.count(), payments_path),
    ],
    ["dataset", "record_count", "path"]
)

display(generation_summary)

print("Diretórios criados no volume:")

for item in dbutils.fs.ls(BASE_PATH):
    print(item.path)